# Universe Selection
- Using all symbols ohlcv1d and volume information
- Based on selection period from 2022-06-10 to 2024-06-10
- 2000 most liquid stocks

In [8]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

UNIVERSE="ALL_SYMBOLS"
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic


In [9]:
# df_bars = (
#     await adapter.get_data(
#         tickers=UNIVERSE,
#         start_date=START,
#         end_date=END,
#         dataset=DATASET,
#         schema="ohlcv-1d",
#     )
# )

In [10]:
# trades with the bbo
df_bars = (
    adapter.scan_data(
        tickers=UNIVERSE,
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d",
    )
    .select(["ts_event", "symbol", "close", "volume"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .collect(engine="streaming")
)

[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


In [11]:
df_bars

ts_event,symbol,close,volume
"datetime[ns, UTC]",cat,f64,u64
2023-06-01 00:00:00 UTC,"""BITE+""",0.06,9996
2023-06-01 00:00:00 UTC,"""MITQ""",1.16,1
2023-06-01 00:00:00 UTC,"""XRAY""",37.61,862389
2023-06-01 00:00:00 UTC,"""DIHP""",23.905,8497
2023-06-01 00:00:00 UTC,"""MOAT""",75.55,40071
…,…,…,…
2025-05-30 00:00:00 UTC,"""PN""",2.86,192
2025-05-30 00:00:00 UTC,"""CASH""",78.05,110712
2025-05-30 00:00:00 UTC,"""CHIQ""",20.94,2911


## Better Universe Selection

In [12]:
# Rank every symbol by median daily dollar volume, then keep the most liquid names.
N_NAMES = 2000
TOTAL_DAYS = df_bars["ts_event"].n_unique()
MIN_DAYS = 0.9 * TOTAL_DAYS      # must have traded most of the window (drops new / delisted names)
MIN_PRICE = 1.0     # drop penny / sub-$5 names

liq = (
    df_bars
    .with_columns((pl.col("close") * pl.col("volume")).alias("dollar_vol"))
    .group_by("symbol")
    .agg(
        pl.col("dollar_vol").median().alias("adv"),   # median is robust to one-off volume spikes
        pl.len().alias("n_days"),
        pl.col("close").last().alias("last_px"),
    )
    .filter(
        (pl.col("n_days") >= MIN_DAYS),
        (pl.col("last_px") >= MIN_PRICE),
    )
    .sort("adv", descending=True)
)

print(f"{liq.height} symbols pass the liquidity filters")
liq.head(N_NAMES)


6547 symbols pass the liquidity filters


symbol,adv,n_days,last_px
cat,f64,u32,f64
"""NVDA""",5.8629e9,501,134.53
"""SPY""",4.5588e9,501,588.39
"""TSLA""",3.7556e9,501,345.5
"""QQQ""",3.7112e9,501,518.37
"""AAPL""",3.1886e9,501,200.29
…,…,…,…
"""EFSC""",2.9118e6,501,52.94
"""TGNA""",2.8950e6,501,16.71
"""FIZZ""",2.8923e6,501,45.23


In [13]:
liq.head(N_NAMES)["adv"].describe()

statistic,value
str,f64
"""count""",2000.0
"""null_count""",0.0
"""mean""",4.6447e7
"""std""",2.4480e8
"""min""",2.8857e6
"""25%""",5.4519e6
"""50%""",1.1168e7
"""75%""",2.7198e7
"""max""",5.8629e9


In [14]:
# Take the top N most liquid names and persist the universe.
# (Swap head(N_NAMES) for a hard ADV threshold, e.g. .filter(pl.col("adv") > 1e6),
#  if you want a count that stays stable across rebuilds.)
universe = liq.head(N_NAMES)["symbol"].cast(pl.Utf8).to_list()

out_path = Path("../universe.txt")
out_path.write_text("\n".join(universe))
print(f"Wrote {len(universe)} symbols to {out_path.resolve()}")
universe[-20:]


Wrote 2000 symbols to /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/universe.txt


['PPH',
 'LCII',
 'AX',
 'FNB',
 'ESGD',
 'SKYY',
 'RITM',
 'SHYG',
 'TREE',
 'AG',
 'QQQE',
 'ATGE',
 'BKH',
 'SPHY',
 'FTSM',
 'EFSC',
 'TGNA',
 'FIZZ',
 'MAC',
 'ANGL']